# Análises finais reproduzíveis — TCC IA e Mercado de Trabalho

Este notebook reproduz as análises finais a partir das bases tratadas disponíveis em `apendices/planilhas` e `apendices/parquet`.

Ele não reexecuta o pipeline completo desde as bases brutas, pois essa etapa envolve arquivos muito pesados, processamento extenso e uso de ferramenta paga de embeddings.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..")
PLANILHAS = ROOT / "apendices" / "planilhas"
PARQUET = ROOT / "apendices" / "parquet"
OUT = ROOT / "resultados"

OUT.mkdir(exist_ok=True)

print("Pasta de planilhas:", PLANILHAS.resolve())
print("Pasta de parquets:", PARQUET.resolve())
print("Pasta de resultados:", OUT.resolve())

## 1. Verificação dos arquivos necessários

In [ ]:
arquivos = {
    "base_ocupacional": PLANILHAS / "cbo_final_com_felten_gmyrek_cod_FINAL_com_fuzzy.xlsx",
    "pnad_analitica": PARQUET / "pnad_2020_2025_analitica_ia.parquet",
    "pnad_felten": PARQUET / "pnad_2020_2025_felten.parquet",
    "pnad_gmyrek": PARQUET / "pnad_2020_2025_gmyrek.parquet",
    "agregado_felten": PARQUET / "agregado_felten_cod_ano.parquet",
    "agregado_gmyrek": PARQUET / "agregado_gmyrek_cod_ano.parquet",
}

for nome, caminho in arquivos.items():
    status = "OK" if caminho.exists() else "FALTANDO"
    print(f"{nome:20s} | {status:8s} | {caminho}")

## 2. Carregamento das bases finais

In [ ]:
base_ocupacional = pd.read_excel(arquivos["base_ocupacional"])
pnad = pd.read_parquet(arquivos["pnad_analitica"])
pnad_felten = pd.read_parquet(arquivos["pnad_felten"])
pnad_gmyrek = pd.read_parquet(arquivos["pnad_gmyrek"])
agregado_felten = pd.read_parquet(arquivos["agregado_felten"])
agregado_gmyrek = pd.read_parquet(arquivos["agregado_gmyrek"])

print("Base ocupacional:", base_ocupacional.shape)
print("PNAD analítica:", pnad.shape)
print("PNAD Felten:", pnad_felten.shape)
print("PNAD Gmyrek:", pnad_gmyrek.shape)
print("Agregado Felten:", agregado_felten.shape)
print("Agregado Gmyrek:", agregado_gmyrek.shape)

## 3. Cobertura dos indicadores na base ocupacional

In [ ]:
indicadores = ["AIOE", "Exposure", "Mean", "SD", "COD_Grupo_Base"]

cobertura = []
for col in indicadores:
    if col in base_ocupacional.columns:
        n = base_ocupacional[col].notna().sum()
        total = len(base_ocupacional)
        cobertura.append({
            "Indicador": col,
            "N preenchido": n,
            "Total": total,
            "Cobertura (%)": round(n / total * 100, 2)
        })

cobertura_df = pd.DataFrame(cobertura)
display(cobertura_df)

cobertura_df.to_csv(OUT / "cobertura_indicadores.csv", index=False)

## 4. Matriz de correlação — AIOE, Mean e SD

In [ ]:
for col in ["AIOE", "Mean", "SD"]:
    if col in base_ocupacional.columns:
        base_ocupacional[col] = pd.to_numeric(base_ocupacional[col], errors="coerce")

corr_cols = [c for c in ["AIOE", "Mean", "SD"] if c in base_ocupacional.columns]
corr = base_ocupacional[corr_cols].corr(method="pearson")

display(corr)

plt.figure(figsize=(6, 5))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlação de Pearson")
plt.xticks(range(len(corr_cols)), corr_cols)
plt.yticks(range(len(corr_cols)), corr_cols)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

plt.title("Matriz de correlação entre indicadores de exposição à IA")
plt.tight_layout()
plt.savefig(OUT / "matriz_correlacao_indicadores.png", dpi=300)
plt.show()

## 5. Distribuição dos indicadores AIOE e Mean

In [ ]:
if "AIOE" in base_ocupacional.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(base_ocupacional["AIOE"].dropna(), bins=30)
    plt.xlabel("AIOE")
    plt.ylabel("Frequência")
    plt.title("Distribuição dos scores AIOE")
    plt.tight_layout()
    plt.savefig(OUT / "distribuicao_aioe.png", dpi=300)
    plt.show()

if "Mean" in base_ocupacional.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(base_ocupacional["Mean"].dropna(), bins=30)
    plt.xlabel("Mean — Gmyrek")
    plt.ylabel("Frequência")
    plt.title("Distribuição do indicador Mean")
    plt.tight_layout()
    plt.savefig(OUT / "distribuicao_gmyrek_mean.png", dpi=300)
    plt.show()

## 6. Exposição à IA versus renda ocupacional

In [ ]:
def scatter_renda_exposicao(df, x_col, titulo, nome_arquivo):
    if x_col not in df.columns or "log_renda_media_ponderada" not in df.columns:
        print(f"Colunas necessárias ausentes para {titulo}")
        return

    temp = df[[x_col, "log_renda_media_ponderada"]].dropna()

    plt.figure(figsize=(8, 5))
    plt.scatter(temp[x_col], temp["log_renda_media_ponderada"], alpha=0.6)

    if len(temp) > 1:
        coef = np.polyfit(temp[x_col], temp["log_renda_media_ponderada"], 1)
        x_line = np.linspace(temp[x_col].min(), temp[x_col].max(), 100)
        y_line = coef[0] * x_line + coef[1]
        plt.plot(x_line, y_line)

    plt.xlabel(x_col)
    plt.ylabel("Log da renda média ponderada")
    plt.title(titulo)
    plt.tight_layout()
    plt.savefig(OUT / nome_arquivo, dpi=300)
    plt.show()

scatter_renda_exposicao(
    agregado_felten,
    "AIOE",
    "Exposição à IA e renda ocupacional — Felten/AIOE",
    "scatter_renda_aioe.png"
)

scatter_renda_exposicao(
    agregado_gmyrek,
    "Mean",
    "Exposição à IA e renda ocupacional — Gmyrek/Mean",
    "scatter_renda_gmyrek.png"
)

## 7. Top 20 e Bottom 20 ocupações

In [ ]:
def ranking_ocupacoes(df, score_col, nome_ocupacao="COD_Titulacao"):
    cols = [c for c in [nome_ocupacao, score_col, "renda_media_ponderada", "peso_total"] if c in df.columns]
    temp = df[cols].dropna(subset=[score_col]).copy()

    # Se houver repetição por trimestre/ano, tira média por ocupação
    agg_dict = {score_col: "mean"}
    if "renda_media_ponderada" in temp.columns:
        agg_dict["renda_media_ponderada"] = "mean"
    if "peso_total" in temp.columns:
        agg_dict["peso_total"] = "sum"

    ranking = temp.groupby(nome_ocupacao, as_index=False).agg(agg_dict)

    top20 = ranking.sort_values(score_col, ascending=False).head(20)
    bottom20 = ranking.sort_values(score_col, ascending=True).head(20)

    return top20, bottom20

top_aioe, bottom_aioe = ranking_ocupacoes(agregado_felten, "AIOE")
top_mean, bottom_mean = ranking_ocupacoes(agregado_gmyrek, "Mean")

display(top_aioe)
display(bottom_aioe)
display(top_mean)
display(bottom_mean)

top_aioe.to_csv(OUT / "top20_aioe.csv", index=False)
bottom_aioe.to_csv(OUT / "bottom20_aioe.csv", index=False)
top_mean.to_csv(OUT / "top20_gmyrek_mean.csv", index=False)
bottom_mean.to_csv(OUT / "bottom20_gmyrek_mean.csv", index=False)

## 8. Evolução temporal da exposição média

In [ ]:
def evolucao_temporal(df, score_col, titulo, nome_arquivo):
    if score_col not in df.columns:
        print(f"Coluna {score_col} ausente")
        return

    cols = [score_col]
    for c in ["Ano", "Trimestre", "peso_total"]:
        if c in df.columns:
            cols.append(c)

    temp = df[cols].dropna(subset=[score_col]).copy()

    if not {"Ano", "Trimestre"}.issubset(temp.columns):
        print("Colunas Ano e Trimestre ausentes")
        return

    if "peso_total" in temp.columns:
        serie = (
            temp.groupby(["Ano", "Trimestre"])
            .apply(lambda x: np.average(x[score_col], weights=x["peso_total"]))
            .reset_index(name=score_col)
        )
    else:
        serie = temp.groupby(["Ano", "Trimestre"], as_index=False)[score_col].mean()

    serie["periodo"] = serie["Ano"].astype(str) + "T" + serie["Trimestre"].astype(str)

    plt.figure(figsize=(10, 5))
    plt.plot(serie["periodo"], serie[score_col], marker="o")
    plt.xticks(rotation=45)
    plt.xlabel("Período")
    plt.ylabel(score_col)
    plt.title(titulo)
    plt.tight_layout()
    plt.savefig(OUT / nome_arquivo, dpi=300)
    plt.show()

    return serie

serie_aioe = evolucao_temporal(
    agregado_felten,
    "AIOE",
    "Evolução trimestral da exposição média — Felten/AIOE",
    "evolucao_temporal_aioe.png"
)

serie_mean = evolucao_temporal(
    agregado_gmyrek,
    "Mean",
    "Evolução trimestral da exposição média — Gmyrek/Mean",
    "evolucao_temporal_gmyrek.png"
)

## 9. Encerramento

Os arquivos gerados pelo notebook são salvos na pasta `resultados/`.